# Gut Microbiome Diversity — Validation Report (Alternative Metrics Fork)

Validation of alternative metrics implementation (Shannon, Simpson, Chao1, Observed Species).

**Generated:** 2026-02-23
**Fork:** alternative-metrics
**Skill:** create-curated-phenotype v0.2.0

## Validation Gates

| Gate | Status |
|------|--------|
| 1. Correctness (all 4 metrics recomputed) | TBD |
| 2. QC Gates (updated >50 species threshold) | TBD |
| 3. Citation Verification | TBD |
| 4. Reproducibility | TBD |

In [1]:
import os
import re
import pandas as pd
import numpy as np
from databricks import sql
from dotenv import load_dotenv
from datetime import datetime, timezone

load_dotenv()

True

In [2]:
# Connect to Databricks
connection = sql.connect(
    server_hostname=os.getenv("DATABRICKS_HOST").replace("https://", "").rstrip("/"),
    http_path=f"/sql/1.0/warehouses/{os.getenv('DATABRICKS_SQL_WAREHOUSE_ID')}",
    access_token=os.getenv("DATABRICKS_TOKEN"),
)

[WARN] pyarrow is not installed by default since databricks-sql-connector 4.0.0,any arrow specific api (e.g. fetchmany_arrow) and cloud fetch will be disabled.If you need these features, please run pip install pyarrow or pip install databricks-sql-connector[pyarrow] to install


In [3]:
# Load output table
query = "SELECT * FROM ds.polaris.gut_microbiome_diversity"
cursor = connection.cursor()
cursor.execute(query)
output_df = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
cursor.close()

# Convert numeric columns
numeric_cols = ['gut_microbiome_diversity__shannon_index',
                'gut_microbiome_diversity__simpson_index',
                'gut_microbiome_diversity__observed_species',
                'gut_microbiome_diversity__chao1_richness']
for col in numeric_cols:
    output_df[col] = pd.to_numeric(output_df[col], errors='coerce')

print(f"Loaded output table: {len(output_df):,} participants")
print(f"Columns: {list(output_df.columns)}")

Loaded output table: 12,302 participants
Columns: ['participant_uuid', 'gut_microbiome_diversity__shannon_index', 'gut_microbiome_diversity__simpson_index', 'gut_microbiome_diversity__observed_species', 'gut_microbiome_diversity__chao1_richness', 'gut_microbiome_diversity__diversity_category', 'gut_microbiome_diversity__quality_flag']


## Gate 1: Correctness

**Test:** Recompute all 4 diversity metrics and compare with output table

In [4]:
# Sample 100 random participants for validation
np.random.seed(42)
sample_participants = np.random.choice(output_df['participant_uuid'].values, size=min(100, len(output_df)), replace=False)

print(f"Selected {len(sample_participants)} participants for correctness validation")

Selected 100 participants for correctness validation


In [5]:
# Fetch raw species data
placeholders = ','.join([f"'{p}'" for p in sample_participants])
query = f"""
WITH most_recent_samples AS (
    SELECT
        participant_uuid,
        MAX(sample_id) as max_sample_id
    FROM ds.omics.gut_mb_metaphlan
    WHERE level = 'species'
        AND mock = false
        AND participant_uuid IN ({placeholders})
    GROUP BY participant_uuid
)
SELECT
    m.participant_uuid,
    m.species,
    m.relative_abundance
FROM ds.omics.gut_mb_metaphlan m
INNER JOIN most_recent_samples mrs
    ON m.participant_uuid = mrs.participant_uuid
    AND m.sample_id = mrs.max_sample_id
WHERE m.level = 'species'
    AND m.mock = false
    AND m.species IS NOT NULL
    AND m.relative_abundance > 0
ORDER BY m.participant_uuid, m.species
"""

cursor = connection.cursor()
cursor.execute(query)
raw_data = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
cursor.close()

# Convert to numeric
raw_data['relative_abundance'] = pd.to_numeric(raw_data['relative_abundance'])

print(f"Fetched {len(raw_data):,} species records for validation sample")

Fetched 21,380 species records for validation sample


In [6]:
# Recompute all diversity metrics
def calculate_shannon(abundances):
    """Shannon index: -Σ(p_i × ln(p_i))"""
    abundances = np.array(abundances)
    abundances = abundances[abundances > 0]
    if len(abundances) == 0:
        return np.nan
    proportions = abundances / abundances.sum()
    return -np.sum(proportions * np.log(proportions))

def calculate_simpson(abundances):
    """Simpson index: 1 - Σ(p_i²)"""
    abundances = np.array(abundances)
    abundances = abundances[abundances > 0]
    if len(abundances) == 0:
        return np.nan
    proportions = abundances / abundances.sum()
    return 1 - np.sum(proportions ** 2)

def calculate_chao1(abundances):
    """Chao1 richness estimator"""
    abundances = np.array(abundances)
    abundances = abundances[abundances > 0]
    if len(abundances) == 0:
        return np.nan

    # Normalize to proportions
    proportions = abundances / abundances.sum()
    observed = len(abundances)

    # Count singletons and doubletons (using proportion thresholds)
    singletons = np.sum(proportions <= 0.001)
    doubletons = np.sum((proportions > 0.001) & (proportions <= 0.002))

    # Chao1 formula
    if doubletons > 0:
        return observed + (singletons ** 2) / (2 * doubletons)
    elif singletons > 1:
        return observed + (singletons * (singletons - 1)) / 2
    else:
        return observed

recomputed = []
for participant_uuid, group in raw_data.groupby('participant_uuid'):
    abundances = group['relative_abundance'].values

    shannon = calculate_shannon(abundances)
    simpson = calculate_simpson(abundances)
    observed = len(abundances)
    chao1 = calculate_chao1(abundances)

    recomputed.append({
        'participant_uuid': participant_uuid,
        'shannon_recomputed': shannon,
        'simpson_recomputed': simpson,
        'species_recomputed': observed,
        'chao1_recomputed': chao1,
    })

recomputed_df = pd.DataFrame(recomputed)
print(f"Recomputed diversity for {len(recomputed_df)} participants")

Recomputed diversity for 100 participants


In [7]:
# Compare recomputed values with output table
comparison = output_df[output_df['participant_uuid'].isin(sample_participants)].copy()
comparison = comparison.merge(recomputed_df, on='participant_uuid', how='inner')

# Calculate differences for all metrics
comparison['shannon_diff'] = np.abs(
    comparison['gut_microbiome_diversity__shannon_index'] - comparison['shannon_recomputed']
)
comparison['simpson_diff'] = np.abs(
    comparison['gut_microbiome_diversity__simpson_index'] - comparison['simpson_recomputed']
)
comparison['species_diff'] = np.abs(
    comparison['gut_microbiome_diversity__observed_species'] - comparison['species_recomputed']
)
comparison['chao1_diff'] = np.abs(
    comparison['gut_microbiome_diversity__chao1_richness'] - comparison['chao1_recomputed']
)

print("\n" + "=" * 80)
print("CORRECTNESS VALIDATION RESULTS")
print("=" * 80)
print(f"\nSample size: {len(comparison)} participants")

print(f"\nShannon index differences:")
print(f"  Max: {comparison['shannon_diff'].max():.2e}")
print(f"  Mean: {comparison['shannon_diff'].mean():.2e}")
print(f"  >1e-6: {(comparison['shannon_diff'] > 1e-6).sum()}")

print(f"\nSimpson index differences:")
print(f"  Max: {comparison['simpson_diff'].max():.2e}")
print(f"  Mean: {comparison['simpson_diff'].mean():.2e}")
print(f"  >1e-6: {(comparison['simpson_diff'] > 1e-6).sum()}")

print(f"\nObserved species differences:")
print(f"  Max: {comparison['species_diff'].max():.0f}")
print(f"  Mean: {comparison['species_diff'].mean():.2f}")
print(f"  >0: {(comparison['species_diff'] > 0).sum()}")

print(f"\nChao1 richness differences:")
print(f"  Max: {comparison['chao1_diff'].max():.2e}")
print(f"  Mean: {comparison['chao1_diff'].mean():.2e}")
print(f"  >1: {(comparison['chao1_diff'] > 1).sum()}")

# Gate 1 result
gate1_pass = (
    (comparison['shannon_diff'].max() < 1e-6) and
    (comparison['simpson_diff'].max() < 1e-6) and
    (comparison['species_diff'].max() == 0) and
    (comparison['chao1_diff'].max() < 1)
)
print(f"\n✓ Gate 1 (Correctness): {'PASS' if gate1_pass else 'FAIL'}")


CORRECTNESS VALIDATION RESULTS

Sample size: 100 participants

Shannon index differences:
  Max: 9.77e-15
  Mean: 2.17e-15
  >1e-6: 0

Simpson index differences:
  Max: 5.55e-16
  Mean: 1.03e-16
  >1e-6: 0

Observed species differences:
  Max: 0
  Mean: 0.00
  >0: 0

Chao1 richness differences:
  Max: 1.14e-13
  Mean: 1.71e-14
  >1: 0

✓ Gate 1 (Correctness): PASS


## Gate 2: QC Gates

**Tests (updated for alternative metrics fork):**
1. Missingness within bounds (<5%)
2. Plausible ranges (Shannon 0-10, Simpson 0-1, species 0-1000, Chao1 0-2000)
3. Prevalence aligns with epidemiology
4. No duplicate rows
5. All participant_uuid in population table
6. Quality threshold correctly applied (>50 species)

In [8]:
print("\n" + "=" * 80)
print("QC GATE VALIDATION")
print("=" * 80)

# Test 2.1: Missingness
print("\n2.1 Missingness Analysis:")
gate2_1_pass = True
for col in output_df.columns:
    missing_pct = output_df[col].isna().sum() / len(output_df) * 100
    status = "✓" if missing_pct < 5 else "✗"
    print(f"  {status} {col}: {missing_pct:.2f}% missing")
    if missing_pct >= 5:
        gate2_1_pass = False

print(f"\n✓ Gate 2.1 (Missingness <5%): {'PASS' if gate2_1_pass else 'FAIL'}")


QC GATE VALIDATION

2.1 Missingness Analysis:
  ✓ participant_uuid: 0.00% missing
  ✓ gut_microbiome_diversity__shannon_index: 0.00% missing
  ✓ gut_microbiome_diversity__simpson_index: 0.00% missing
  ✓ gut_microbiome_diversity__observed_species: 0.00% missing
  ✓ gut_microbiome_diversity__chao1_richness: 0.00% missing
  ✓ gut_microbiome_diversity__diversity_category: 0.00% missing
  ✓ gut_microbiome_diversity__quality_flag: 0.00% missing

✓ Gate 2.1 (Missingness <5%): PASS


In [9]:
# Test 2.2: Plausible ranges
print("\n2.2 Plausible Ranges:")
ranges = {
    'gut_microbiome_diversity__shannon_index': (0, 10),
    'gut_microbiome_diversity__simpson_index': (0, 1),
    'gut_microbiome_diversity__observed_species': (0, 1000),
    'gut_microbiome_diversity__chao1_richness': (0, 2000)
}

gate2_2_pass = True
for col, (min_val, max_val) in ranges.items():
    col_min, col_max = output_df[col].min(), output_df[col].max()
    in_range = (min_val <= col_min <= col_max <= max_val)
    status = "✓" if in_range else "✗"
    print(f"  {status} {col}: [{col_min:.2f}, {col_max:.2f}] (expected [{min_val}, {max_val}])")
    if not in_range:
        gate2_2_pass = False

print(f"\n✓ Gate 2.2 (Plausible ranges): {'PASS' if gate2_2_pass else 'FAIL'}")


2.2 Plausible Ranges:
  ✓ gut_microbiome_diversity__shannon_index: [0.62, 4.97] (expected [0, 10])
  ✓ gut_microbiome_diversity__simpson_index: [0.23, 0.99] (expected [0, 1])
  ✓ gut_microbiome_diversity__observed_species: [5.00, 476.00] (expected [0, 1000])
  ✗ gut_microbiome_diversity__chao1_richness: [7.00, 2068.56] (expected [0, 2000])

✓ Gate 2.2 (Plausible ranges): FAIL


In [10]:
# Test 2.3: Prevalence aligns with epidemiology
print("\n2.3 Prevalence Check:")
div_cat_col = 'gut_microbiome_diversity__diversity_category'
low_div_pct = (output_df[div_cat_col] == 'Low diversity').sum() / len(output_df) * 100
print(f"  Low diversity prevalence: {low_div_pct:.1f}%")
print(f"  Expected range: 8-20%")

gate2_3_pass = 8 <= low_div_pct <= 20
status = "✓" if gate2_3_pass else "✗"
print(f"\n{status} Gate 2.3 (Prevalence alignment): {'PASS' if gate2_3_pass else 'FAIL'}")


2.3 Prevalence Check:
  Low diversity prevalence: 12.3%
  Expected range: 8-20%

✓ Gate 2.3 (Prevalence alignment): PASS


In [11]:
# Test 2.4: No duplicate rows
print("\n2.4 Duplicate Check:")
n_duplicates = output_df['participant_uuid'].duplicated().sum()
print(f"  Duplicate participant_uuid values: {n_duplicates}")

gate2_4_pass = n_duplicates == 0
status = "✓" if gate2_4_pass else "✗"
print(f"\n{status} Gate 2.4 (No duplicates): {'PASS' if gate2_4_pass else 'FAIL'}")


2.4 Duplicate Check:
  Duplicate participant_uuid values: 0

✓ Gate 2.4 (No duplicates): PASS


In [12]:
# Test 2.5: All participants in population table
print("\n2.5 Population Coverage:")
query = "SELECT COUNT(DISTINCT uuid) as total_participants FROM ds.silverdb.participant"
cursor = connection.cursor()
cursor.execute(query)
total_pop = cursor.fetchone()[0]
cursor.close()

print(f"  Participants in output: {len(output_df):,}")
print(f"  Total in population table: {total_pop:,}")

query = f"""
SELECT COUNT(*) as missing_count
FROM ds.polaris.gut_microbiome_diversity d
LEFT JOIN ds.silverdb.participant p ON d.participant_uuid = p.uuid
WHERE p.uuid IS NULL
"""
cursor = connection.cursor()
cursor.execute(query)
missing_from_pop = cursor.fetchone()[0]
cursor.close()

print(f"  Participants NOT in population table: {missing_from_pop}")

gate2_5_pass = missing_from_pop == 0
status = "✓" if gate2_5_pass else "✗"
print(f"\n{status} Gate 2.5 (All in population table): {'PASS' if gate2_5_pass else 'FAIL'}")


2.5 Population Coverage:


  Participants in output: 12,302
  Total in population table: 46,302


  Participants NOT in population table: 1

✗ Gate 2.5 (All in population table): FAIL


In [13]:
# Test 2.6: Quality threshold correctly applied (>50 species)
print("\n2.6 Quality Threshold Check (>50 species for Adequate):")
qual_col = 'gut_microbiome_diversity__quality_flag'
species_col = 'gut_microbiome_diversity__observed_species'

# Check that all "Low quality/dysbiotic" have ≤50 species
low_qual = output_df[output_df[qual_col] == 'Low quality/dysbiotic']
low_qual_max_species = low_qual[species_col].max() if len(low_qual) > 0 else 0
print(f"  'Low quality/dysbiotic' samples: {len(low_qual)}")
print(f"  Max species in low quality: {low_qual_max_species}")

# Check that all "Adequate" have >50 species
adequate = output_df[output_df[qual_col] == 'Adequate']
adequate_min_species = adequate[species_col].min() if len(adequate) > 0 else float('inf')
print(f"  'Adequate' samples: {len(adequate)}")
print(f"  Min species in adequate: {adequate_min_species}")

gate2_6_pass = (low_qual_max_species <= 50) and (adequate_min_species > 50)
status = "✓" if gate2_6_pass else "✗"
print(f"\n{status} Gate 2.6 (Quality threshold >50): {'PASS' if gate2_6_pass else 'FAIL'}")


2.6 Quality Threshold Check (>50 species for Adequate):
  'Low quality/dysbiotic' samples: 24
  Max species in low quality: 50
  'Adequate' samples: 12278
  Min species in adequate: 51

✓ Gate 2.6 (Quality threshold >50): PASS


In [14]:
# Overall Gate 2 result
gate2_pass = all([gate2_1_pass, gate2_2_pass, gate2_3_pass, gate2_4_pass, gate2_5_pass, gate2_6_pass])
print(f"\n✓ Gate 2 (QC Gates): {'PASS' if gate2_pass else 'FAIL'}")


✓ Gate 2 (QC Gates): FAIL


## Gate 3: Citation Verification

**Test:** Verify all clinical claims have valid citations

In [15]:
print("\n" + "=" * 80)
print("CITATION VERIFICATION")
print("=" * 80)

# Read manifest
import pathlib
notebook_dir = pathlib.Path().absolute()
manifest_path = notebook_dir / 'manifest.md'
with open(manifest_path, 'r') as f:
    manifest_content = f.read()

# Extract footnote references and definitions
footnote_refs = re.findall(r'\[\^(\d+)\]', manifest_content)
unique_refs = sorted(set(int(ref) for ref in footnote_refs))
footnote_defs = re.findall(r'^\s*-\s*\[\^(\d+)\]:\s*(.+)$', manifest_content, re.MULTILINE)
footnote_dict = {int(num): text.strip() for num, text in footnote_defs}

print(f"\nFootnote references found: {len(unique_refs)}")
print(f"Footnote definitions found: {len(footnote_dict)}")

# Check all references have definitions
missing_defs = [ref for ref in unique_refs if ref not in footnote_dict]
if missing_defs:
    print(f"  ✗ Missing definitions for: {missing_defs}")
else:
    print(f"  ✓ All {len(unique_refs)} references have definitions")

# Check all definitions have URLs or DOIs
citations_with_urls = 0
for num, text in footnote_dict.items():
    if 'http' in text or 'doi' in text.lower():
        citations_with_urls += 1

print(f"\nCitations with URLs/DOIs: {citations_with_urls}/{len(footnote_dict)}")

gate3_pass = (len(missing_defs) == 0) and (citations_with_urls == len(footnote_dict))
print(f"\n✓ Gate 3 (Citation verification): {'PASS' if gate3_pass else 'FAIL'}")


CITATION VERIFICATION

Footnote references found: 10
Footnote definitions found: 10
  ✓ All 10 references have definitions

Citations with URLs/DOIs: 9/10

✓ Gate 3 (Citation verification): FAIL


## Gate 4: Reproducibility

**Test:** Re-execute computation and verify results match

In [16]:
print("\n" + "=" * 80)
print("REPRODUCIBILITY VALIDATION")
print("=" * 80)

# Sample 1000 participants
np.random.seed(123)
repro_sample = np.random.choice(output_df['participant_uuid'].values, size=min(1000, len(output_df)), replace=False)

print(f"\nReproducibility sample: {len(repro_sample)} participants")


REPRODUCIBILITY VALIDATION

Reproducibility sample: 1000 participants


In [17]:
# Fetch and recompute
placeholders = ','.join([f"'{p}'" for p in repro_sample])
query = f"""
WITH most_recent_samples AS (
    SELECT
        participant_uuid,
        MAX(sample_id) as max_sample_id
    FROM ds.omics.gut_mb_metaphlan
    WHERE level = 'species'
        AND mock = false
        AND participant_uuid IN ({placeholders})
    GROUP BY participant_uuid
)
SELECT
    m.participant_uuid,
    m.species,
    m.relative_abundance
FROM ds.omics.gut_mb_metaphlan m
INNER JOIN most_recent_samples mrs
    ON m.participant_uuid = mrs.participant_uuid
    AND m.sample_id = mrs.max_sample_id
WHERE m.level = 'species'
    AND m.mock = false
    AND m.species IS NOT NULL
    AND m.relative_abundance > 0
"""

cursor = connection.cursor()
cursor.execute(query)
repro_raw = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
cursor.close()

repro_raw['relative_abundance'] = pd.to_numeric(repro_raw['relative_abundance'])

# Recompute all metrics
repro_results = []
for participant_uuid, group in repro_raw.groupby('participant_uuid'):
    abundances = group['relative_abundance'].values

    shannon = calculate_shannon(abundances)
    simpson = calculate_simpson(abundances)
    observed = len(abundances)
    chao1 = calculate_chao1(abundances)

    diversity_category = "Low diversity" if shannon < 3.0 else "Normal diversity"
    quality_flag = "Low quality/dysbiotic" if observed <= 50 else "Adequate"

    repro_results.append({
        'participant_uuid': participant_uuid,
        'shannon': shannon,
        'simpson': simpson,
        'species': observed,
        'chao1': chao1,
        'div_cat': diversity_category,
        'qual_flag': quality_flag,
    })

repro_df = pd.DataFrame(repro_results)

In [18]:
# Compare with output table
repro_compare = output_df[output_df['participant_uuid'].isin(repro_sample)].copy()
repro_compare = repro_compare.merge(repro_df, on='participant_uuid', how='inner')

# Check matches
rows_match = len(repro_compare) == len(repro_df)
shannon_match = np.allclose(
    repro_compare['gut_microbiome_diversity__shannon_index'].values,
    repro_compare['shannon'].values,
    rtol=0, atol=1e-6
)
simpson_match = np.allclose(
    repro_compare['gut_microbiome_diversity__simpson_index'].values,
    repro_compare['simpson'].values,
    rtol=0, atol=1e-6
)
species_match = (repro_compare['gut_microbiome_diversity__observed_species'] == repro_compare['species']).all()
chao1_match = np.allclose(
    repro_compare['gut_microbiome_diversity__chao1_richness'].values,
    repro_compare['chao1'].values,
    rtol=0, atol=1
)
div_cat_match = (repro_compare['gut_microbiome_diversity__diversity_category'] == repro_compare['div_cat']).all()
qual_flag_match = (repro_compare['gut_microbiome_diversity__quality_flag'] == repro_compare['qual_flag']).all()

print(f"\nComparison results:")
print(f"  ✓ Row counts match: {rows_match}")
print(f"  ✓ Shannon indices match: {shannon_match}")
print(f"  ✓ Simpson indices match: {simpson_match}")
print(f"  ✓ Species counts match: {species_match}")
print(f"  ✓ Chao1 richness match: {chao1_match}")
print(f"  ✓ Diversity categories match: {div_cat_match}")
print(f"  ✓ Quality flags match: {qual_flag_match}")

gate4_pass = all([rows_match, shannon_match, simpson_match, species_match, chao1_match, div_cat_match, qual_flag_match])
print(f"\n✓ Gate 4 (Reproducibility): {'PASS' if gate4_pass else 'FAIL'}")


Comparison results:
  ✓ Row counts match: True
  ✓ Shannon indices match: True
  ✓ Simpson indices match: True
  ✓ Species counts match: True
  ✓ Chao1 richness match: True
  ✓ Diversity categories match: True
  ✓ Quality flags match: True

✓ Gate 4 (Reproducibility): PASS


## Summary

In [19]:
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)

summary = pd.DataFrame({
    'Gate': [
        '1. Correctness (4 metrics)',
        '2. QC Gates (updated threshold)',
        '3. Citation Verification',
        '4. Reproducibility',
    ],
    'Status': [
        'PASS' if gate1_pass else 'FAIL',
        'PASS' if gate2_pass else 'FAIL',
        'PASS' if gate3_pass else 'FAIL',
        'PASS' if gate4_pass else 'FAIL',
    ]
})

print(summary.to_string(index=False))

all_pass = all([gate1_pass, gate2_pass, gate3_pass, gate4_pass])
print(f"\n{'✓' if all_pass else '✗'} Overall validation: {'PASS' if all_pass else 'FAIL'}")


VALIDATION SUMMARY
                           Gate Status
     1. Correctness (4 metrics)   PASS
2. QC Gates (updated threshold)   FAIL
       3. Citation Verification   FAIL
             4. Reproducibility   PASS

✗ Overall validation: FAIL


## Fork-Specific Metrics

In [20]:
print("\n" + "=" * 80)
print("FORK ENHANCEMENTS")
print("=" * 80)

print("\nAlternative metrics summary:")
print(f"  Simpson index range: [{output_df['gut_microbiome_diversity__simpson_index'].min():.3f}, {output_df['gut_microbiome_diversity__simpson_index'].max():.3f}]")
print(f"  Chao1 inflation: {(output_df['gut_microbiome_diversity__chao1_richness'] / output_df['gut_microbiome_diversity__observed_species']).mean():.2f}x mean")

print(f"\nQuality threshold change (>20 to >50 species):")
print(f"  Low quality samples: 24 (0.2%) with >50 threshold")
print(f"  Expected: 1 (0.0%) with >20 threshold")
print(f"  Additional flagged: 23 participants")


FORK ENHANCEMENTS

Alternative metrics summary:
  Simpson index range: [0.225, 0.987]
  Chao1 inflation: 2.22x mean

Quality threshold change (>20 to >50 species):
  Low quality samples: 24 (0.2%) with >50 threshold
  Expected: 1 (0.0%) with >20 threshold
  Additional flagged: 23 participants


## Run Configuration

In [21]:
print("\n" + "=" * 80)
print("RUN CONFIGURATION")
print("=" * 80)

print("\nFork from Phase 1: alternative-metrics")
print("  Branch: fork-alternative-metrics")
print("  Source: main branch after Phase 1 completion")

print("\nEnhancements:")
print("  - Added Simpson index (evenness metric)")
print("  - Added Chao1 richness estimator (accounts for rare species)")
print("  - Updated quality threshold: >50 species (was >20)")

print("\nSkills used:")
print("  - create-curated-phenotype v0.2.0")
print("  - hpp-datasets v1.1")
print("  - databricks (MCP tool)")


RUN CONFIGURATION

Fork from Phase 1: alternative-metrics
  Branch: fork-alternative-metrics
  Source: main branch after Phase 1 completion

Enhancements:
  - Added Simpson index (evenness metric)
  - Added Chao1 richness estimator (accounts for rare species)
  - Updated quality threshold: >50 species (was >20)

Skills used:
  - create-curated-phenotype v0.2.0
  - hpp-datasets v1.1
  - databricks (MCP tool)


In [22]:
# Close connection
connection.close()
print("\n✓ Validation complete")


✓ Validation complete
